# 034 · SGD with Momentum

Momentum is **lesson 033's EWMA applied to the gradient**. That is the whole
idea:

$$v_t = \beta v_{t-1} + \partial L/\partial w, \qquad w \leftarrow w - \eta v_t$$

Set β = 0 and you get plain gradient descent back.

| Part | What we reproduce |
|---|---|
| A | the ravine, and why the demo is **not rigged** |
| B | plain GD **never** reaches loss < 0.1; momentum reaches it at **step 35** |
| C | consistent directions accumulate, alternating ones cancel |
| D | sweeping β from 0 to 0.99 — where overshoot appears |

Needs `numpy`.

In [ ]:
import numpy as np

# The ravine: L = 0.5(x^2 + 20 y^2). The y axis is 20x steeper than the x axis.
A, B = 1.0, 20.0
LR, STEPS, START = 0.02, 60, (9.0, 1.0)

def grad(p):
    return np.array([A * p[0], B * p[1]])

def loss(p):
    return 0.5 * (A * p[0] ** 2 + B * p[1] ** 2)

print(f"start {START}, lr = {LR}, {STEPS} steps")
print(f"the steep axis diverges above lr = {2/B}, so the rate must stay small")

## Part A — Why the ravine is a fair test, not a rigged one

The obvious objection is that the learning rate was chosen to make plain
gradient descent look bad. It was not. The steep axis **caps** it: above
`2/20 = 0.1` the y direction diverges outright. And that same cap is what
starves the shallow x direction.

In [ ]:
def plain(lr, steps=STEPS):
    p = np.array(START, float)
    for _ in range(steps):
        p = p - lr * grad(p)
    return p

print(f"{'lr':>7}{'final x':>12}{'final y':>14}{'verdict':>12}")
for lr in (0.02, 0.05, 0.09, 0.10, 0.11):
    p = plain(lr)
    bad = (not np.all(np.isfinite(p))) or np.abs(p).max() > 1e3
    print(f"{lr:>7}{p[0]:>12.4f}{p[1]:>14.2e}{'DIVERGED' if bad else 'stable':>12}")

print("\nThe rate cannot go above 0.1 without the steep axis blowing up.")
print("At the largest safe rate the shallow axis is still nowhere near zero.")
print("That is the trap, and it is a property of the surface, not the setup.")

## Part B — Plain gradient descent against momentum

In [ ]:
def descend(momentum, steps=STEPS, lr=LR):
    p, v, path = np.array(START, float), np.zeros(2), []
    for _ in range(steps):
        path.append(p.copy())
        v = momentum * v + grad(p)          # EWMA-ish accumulation of gradient
        p = p - lr * v
    return path


for m, label in ((0.0, "plain GD"), (0.9, "momentum")):
    path = descend(m)
    final = path[-1]
    hit = next((i for i, q in enumerate(path) if loss(q) < 0.1), None)
    print(f"  {label:<10} x={final[0]:7.4f}  loss={loss(final):9.5f}  "
          f"reached loss<0.1 at step {hit if hit is not None else 'never'}")

In [ ]:
plain_path, mom_path = descend(0.0), descend(0.9)
print(f"final loss, plain GD : {loss(plain_path[-1]):.5f}")
print(f"final loss, momentum : {loss(mom_path[-1]):.5f}")
print(f"ratio                : {loss(plain_path[-1]) / loss(mom_path[-1]):,.0f}x lower")

assert next((i for i, q in enumerate(plain_path) if loss(q) < 0.1), None) is None
assert next(i for i, q in enumerate(mom_path) if loss(q) < 0.1) == 35

## Part C — Why it works: accumulate or cancel

One running average does two useful things at once, depending on whether the
gradient keeps pointing the same way.

In [ ]:
def velocity_trace(gradients, beta=0.9):
    v, out = 0.0, []
    for g in gradients:
        v = beta * v + g
        out.append(v)
    return np.array(out)


steady = velocity_trace([1.0] * 20)                    # shallow axis: same sign
alternating = velocity_trace([1.0, -1.0] * 10)         # steep axis: flip-flops

print(f"{'step':>5}{'consistent g=+1':>18}{'alternating g=+/-1':>21}")
for i in (0, 1, 2, 5, 10, 19):
    print(f"{i+1:>5}{steady[i]:>18.3f}{alternating[i]:>21.3f}")

print(f"\nconsistent  -> velocity grows towards 1/(1-beta) = {1/(1-0.9):.0f}x the gradient")
print(f"alternating -> velocity stays near {np.abs(alternating[-6:]).max():.3f}, it cancels itself")
print("\nThe shallow axis is amplified 10x. The steep, oscillating axis is damped.")
print("Same formula, opposite effect, decided entirely by the data.")

In [ ]:
# Confirm the amplification on the real problem: how far along x does each go?
print(f"{'beta':>7}{'x after 60 steps':>20}")
for b in (0.0, 0.5, 0.9):
    print(f"{b:>7}{descend(b)[-1][0]:>20.4f}")
print("\n(x starts at 9.0 and should reach 0.)")

## Part D — Sweeping β, and where overshoot appears

Momentum's gift and its flaw are the same thing: velocity carries you forward.
Across plateaus, that is what you want. Near the minimum, it is not.

In [ ]:
print(f"{'beta':>7}{'final loss':>14}{'step to <0.1':>15}{'max |x| after step 20':>24}")
for b in (0.0, 0.5, 0.9, 0.95, 0.99):
    path = descend(b, steps=120)
    hit = next((i for i, q in enumerate(path) if loss(q) < 0.1), None)
    late = max(abs(q[0]) for q in path[20:])
    print(f"{b:>7}{loss(path[-1]):>14.5f}{str(hit if hit is not None else 'never'):>15}"
          f"{late:>24.4f}")

print("\nUp to 0.9, higher beta arrives sooner. At 0.99 the velocity is so")
print("large that it sails past the minimum and takes a long time to settle.")

In [ ]:
# Overshoot, isolated: a one-dimensional bowl L = 0.5 w^2 from w = 10.
def bowl(beta, lr=0.1, steps=45, w0=10.0):
    w, v, path = w0, 0.0, []
    for _ in range(steps):
        path.append(w)
        v = beta * v + w
        w = w - lr * v
    return np.array(path)

print(f"{'beta':>7}{'crosses zero to':>18}")
for b in (0.0, 0.5, 0.9, 0.95):
    print(f"{b:>7}{bowl(b).min():>18.3f}")

print("\nAt beta = 0 there is no overshoot at all. The same velocity that")
print("crosses plateaus is what carries you past the target.")
print("Lesson 035 fixes exactly this, and costs nothing to do so.")
assert bowl(0.9).min() < -5

## What to take away

- **Momentum is EWMA applied to the gradient** — lesson 033's formula with
  `θ_t = ∂L/∂w`.
- **`v_t = βv_{t−1} + ∂L/∂w`, then `w ← w − ηv_t`.** β = 0 recovers plain
  gradient descent.
- **Consistent directions accumulate; alternating ones cancel** — one running
  average does both.
- Measured on the ravine: plain GD **never reached loss < 0.1** in 60 steps;
  momentum reached it at **step 35** and finished about **1,300× lower**.
- The demo is not rigged: the **steep axis caps the learning rate**, and that
  same cap starves the shallow one.
- **Fixes oscillation and plateaus**, and can sometimes escape shallow local
  minima.
- **Does not fix the single-learning-rate problem** — that is AdaGrad and RMSprop.
- **It overshoots the minimum.** NAG addresses this.
- **Cost: one extra stored value per parameter.** Typical β = 0.9.

## Exercises

1. Change `B` from 20 to 5 and to 100. How does the advantage of momentum scale
   with the ratio of curvatures? Plot it.
2. The velocity converges to `g/(1−β)` for a constant gradient. Derive that, then
   check it against Part C.
3. Momentum is sometimes written `v = βv + (1−β)g` instead. Implement both and
   find the learning rate that makes them equivalent. Which does Keras use?
4. Try momentum on lesson 032's saddle point `L = x² − y²`. Does the velocity
   term help it escape? Compare against plain gradient descent from the same start.
5. Find the β that minimises the step at which the ravine reaches loss < 0.1.
   Is it 0.9? What does that say about the usual default?